# ComfyUI Colab (r2)

r1からの主な改善点:
- ✅ `simpleeval` / `pyopengl` 追加 → nodes_math / nodes_glsl エラー解消
- ✅ torch 再インストール行を削除 → Colab既存の cu128 版をそのまま利用
- ✅ `comfy_kitchen` 追加 → fp8/fp4 量子化対応
- ✅ Google Drive マウントを `USE_GOOGLE_DRIVE` フラグと統合
- ✅ モデルダウンロードを `huggingface_hub` に変更 → リトライ・整合性チェック付き
- ✅ xvfb 仮想ディスプレイ設定 → GLSLノード有効化

## Step 1: Environment Setup
ComfyUI のセットアップ、依存パッケージのインストールを行います。

In [1]:
#@title Environment Setup

from pathlib import Path

OPTIONS = {}

USE_GOOGLE_DRIVE = False  #@param {type:"boolean"} Falseにすると100GBエリアへのインストールに切り替わります
UPDATE_COMFY_UI = True   #@param {type:"boolean"}
USE_COMFYUI_MANAGER = True  #@param {type:"boolean"}
INSTALL_CUSTOM_NODES_DEPENDENCIES = True  #@param {type:"boolean"}

OPTIONS['USE_GOOGLE_DRIVE'] = USE_GOOGLE_DRIVE
OPTIONS['UPDATE_COMFY_UI'] = UPDATE_COMFY_UI
OPTIONS['USE_COMFYUI_MANAGER'] = USE_COMFYUI_MANAGER
OPTIONS['INSTALL_CUSTOM_NODES_DEPENDENCIES'] = INSTALL_CUSTOM_NODES_DEPENDENCIES

current_dir = !pwd
WORKSPACE = f"{current_dir[0]}/ComfyUI"

# ── Google Drive 連動（USE_GOOGLE_DRIVE フラグで制御）──
if OPTIONS['USE_GOOGLE_DRIVE']:
    !echo "Mounting Google Drive..."
    %cd /
    from google.colab import drive
    drive.mount('/content/drive')
    WORKSPACE = "/content/drive/MyDrive/ComfyUI"
    %cd /content/drive/MyDrive
    print(f"✅ Google Drive モード: 画像は {WORKSPACE}/output に保存されます")
else:
    print(f"✅ ローカルモード: 画像は {WORKSPACE}/output に保存されます")

# ── ComfyUI のクローン / 更新 ──
![ ! -d $WORKSPACE ] && echo -= Initial setup ComfyUI =- && git clone https://github.com/comfyanonymous/ComfyUI
%cd $WORKSPACE

if OPTIONS['UPDATE_COMFY_UI']:
    !echo -= Updating ComfyUI =-
    ![ -f ".ci/nightly/update_windows/update_comfyui_and_python_dependencies.bat" ] && chmod 755 .ci/nightly/update_windows/update_comfyui_and_python_dependencies.bat
    ![ -f ".ci/nightly/windows_base_files/run_nvidia_gpu.bat" ] && chmod 755 .ci/nightly/windows_base_files/run_nvidia_gpu.bat
    ![ -f ".ci/update_windows/update_comfyui_and_python_dependencies.bat" ] && chmod 755 .ci/update_windows/update_comfyui_and_python_dependencies.bat
    ![ -f ".ci/update_windows_cu118/update_comfyui_and_python_dependencies.bat" ] && chmod 755 .ci/update_windows_cu118/update_comfyui_and_python_dependencies.bat
    ![ -f ".ci/update_windows/update.py" ] && chmod 755 .ci/update_windows/update.py
    ![ -f ".ci/update_windows/update_comfyui.bat" ] && chmod 755 .ci/update_windows/update_comfyui.bat
    ![ -f ".ci/update_windows/README_VERY_IMPORTANT.txt" ] && chmod 755 .ci/update_windows/README_VERY_IMPORTANT.txt
    ![ -f ".ci/update_windows/run_cpu.bat" ] && chmod 755 .ci/update_windows/run_cpu.bat
    ![ -f ".ci/update_windows/run_nvidia_gpu.bat" ] && chmod 755 .ci/update_windows/run_nvidia_gpu.bat
    !git pull

# ── 依存パッケージ ──
!echo -= Install dependencies =-
!pip install -q accelerate
!pip install -q einops transformers>=4.28.1 safetensors>=0.4.2 aiohttp pyyaml Pillow scipy tqdm psutil tokenizers>=0.13.3

# 【r2改善①】torch は Colab 既存の cu128 版をそのまま使う（再インストール不要）
# !pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121  ← 削除
print("ℹ️  torch: Colab 既存ビルドを使用 (再インストールをスキップ)")
import torch; print(f"   torch version: {torch.__version__}")

!pip install -q torchsde
!pip install -q kornia>=0.7.1 spandrel soundfile sentencepiece
!pip install -q comfyui-workflow-templates
!pip install -q comfyui-embedded-docs

# 【r2改善②】nodes_math / nodes_glsl エラーを解消する追加パッケージ
!pip install -q simpleeval
!pip install -q pyopengl

# 【r2改善③】fp8/fp4 量子化対応
!pip install -q comfy-kitchen

# ── ComfyUI-Manager ──
if OPTIONS['USE_COMFYUI_MANAGER']:
    %cd custom_nodes
    ![ -f "ComfyUI-Manager/check.sh" ] && chmod 755 ComfyUI-Manager/check.sh
    ![ -f "ComfyUI-Manager/scan.sh" ] && chmod 755 ComfyUI-Manager/scan.sh
    ![ -f "ComfyUI-Manager/node_db/dev/scan.sh" ] && chmod 755 ComfyUI-Manager/node_db/dev/scan.sh
    ![ -f "ComfyUI-Manager/node_db/tutorial/scan.sh" ] && chmod 755 ComfyUI-Manager/node_db/tutorial/scan.sh
    ![ -f "ComfyUI-Manager/scripts/install-comfyui-venv-linux.sh" ] && chmod 755 ComfyUI-Manager/scripts/install-comfyui-venv-linux.sh
    ![ -f "ComfyUI-Manager/scripts/install-comfyui-venv-win.bat" ] && chmod 755 ComfyUI-Manager/scripts/install-comfyui-venv-win.bat
    ![ ! -d ComfyUI-Manager ] && echo -= Initial setup ComfyUI-Manager =- && git clone https://github.com/ltdrdata/ComfyUI-Manager
    %cd ComfyUI-Manager
    !git pull

# ComfyUI-Impact-Pack
![ ! -d $WORKSPACE/custom_nodes/ComfyUI-Impact-Pack ] && git clone https://github.com/ltdrdata/ComfyUI-Impact-Pack.git $WORKSPACE/custom_nodes/ComfyUI-Impact-Pack

%cd $WORKSPACE

if OPTIONS['INSTALL_CUSTOM_NODES_DEPENDENCIES']:
    !echo -= Install custom nodes dependencies =-
    !pip install -q GitPython
    !python custom_nodes/ComfyUI-Manager/cm-cli.py restore-dependencies

# rembg / onnxruntime / insightface
!pip install -q rembg onnxruntime insightface

# av / comfy_aimdo
!pip install -q av comfy_aimdo

print("\n✅ Step 1 完了")

✅ ローカルモード: 画像は /content/ComfyUI/output に保存されます
-= Initial setup ComfyUI =-
Cloning into 'ComfyUI'...
remote: Enumerating objects: 35809, done.
remote: Counting objects: 100% (31/31), done.
remote: Compressing objects: 100% (17/17), done.
remote: Total 35809 (delta 18), reused 14 (delta 14), pack-reused 35778 (from 3)
Receiving objects: 100% (35809/35809), 84.21 MiB | 15.49 MiB/s, done.
Resolving deltas: 100% (24296/24296), done.
/content/ComfyUI
-= Updating ComfyUI =-
Already up to date.
-= Install dependencies =-
ℹ️  torch: Colab 既存ビルドを使用 (再インストールをスキップ)
   torch version: 2.10.0+cu128
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.2/61.2 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.8/78.8 MB 10.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.3/73.3 MB 11.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 38.0/38.0 MB 63.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━

## Step 2: 仮想ディスプレイ設定（GLSLノード有効化）
Colab はヘッドレス環境のため、OpenGL を必要とする `nodes_glsl.py` を使うには xvfb が必要です。

In [2]:
#@title 【r2改善④】xvfb 仮想ディスプレイのセットアップ（GLSLノード有効化）

!apt-get install -y -q xvfb
import subprocess, os, time

# 既存の :99 プロセスがあれば終了
subprocess.run(["pkill", "-f", "Xvfb :99"], capture_output=True)
time.sleep(0.5)

# 仮想ディスプレイ起動
subprocess.Popen(["Xvfb", ":99", "-screen", "0", "1024x768x24"])
os.environ['DISPLAY'] = ':99'
time.sleep(1)

print("✅ 仮想ディスプレイ :99 を起動しました (DISPLAY=:99)")

Reading package lists...
Building dependency tree...
Reading state information...
xvfb is already the newest version (2:21.1.4-2ubuntu1.7~22.04.16).
0 upgraded, 0 newly installed, 0 to remove and 42 not upgraded.
✅ 仮想ディスプレイ :99 を起動しました (DISPLAY=:99)


## Step 3: 出力フォルダの設定
生成画像を常に Google Drive の  へ保存します。


In [3]:
#@title 【r2改善⑤】出力先設定（常に Google Drive へ保存）

import os, shutil
from google.colab import drive

# Google Drive マウント
drive.mount("/content/drive", force_remount=False)

# WORKSPACE を再定義（Step1 未実行でも動くよう独立させる）
_workspace = "/content/ComfyUI"

# Drive 側の出力フォルダを作成
drive_output = "/content/drive/MyDrive/ComfyUI_Output"
os.makedirs(drive_output, exist_ok=True)

# ComfyUI の output をシンボリックリンクで差し替え
local_output = f"{_workspace}/output"
if os.path.islink(local_output):
    os.unlink(local_output)
elif os.path.isdir(local_output):
    shutil.rmtree(local_output)

os.symlink(drive_output, local_output)

# 確認
assert os.path.islink(local_output)
assert os.path.exists(local_output)
print("output ->", os.path.realpath(local_output))
print("setup complete: images will be saved to MyDrive/ComfyUI_Output")


Mounted at /content/drive
output -> /content/drive/MyDrive/ComfyUI_Output
setup complete: images will be saved to MyDrive/ComfyUI_Output


## Step 4: モデルのダウンロード
`huggingface_hub` を使ってリトライ付き・整合性チェック付きでダウンロードします。
追加したいモデルはコメントアウトを外してください。

In [4]:
#@title 【r2改善⑥】Anima モデルのダウンロード（パス修正版）

from huggingface_hub import hf_hub_download
import os, shutil, glob

REPO_ID = "circlestone-labs/Anima"
MODEL_BASE = f"{WORKSPACE}/models"

def download_model(repo_id, hf_filename, local_dir):
    """
    hf_hub_download は filename のサブディレクトリ構造をそのまま再現するため、
    ダウンロード後に目的のフォルダへ移動する。
    """
    os.makedirs(local_dir, exist_ok=True)
    basename = os.path.basename(hf_filename)
    dest = os.path.join(local_dir, basename)
    if os.path.exists(dest):
        print(f"skip (exists): {basename}")
        return
    print(f"downloading: {basename} ...")
    tmp_dir = f"{WORKSPACE}/_hf_tmp"
    downloaded_path = hf_hub_download(
        repo_id=repo_id,
        filename=hf_filename,
        local_dir=tmp_dir,
    )
    shutil.move(downloaded_path, dest)
    shutil.rmtree(tmp_dir, ignore_errors=True)
    print(f"done: {basename} -> {dest}")

# Anima 必須モデル
download_model(REPO_ID,
    "split_files/diffusion_models/anima-preview.safetensors",
    f"{MODEL_BASE}/diffusion_models")

download_model(REPO_ID,
    "split_files/text_encoders/qwen_3_06b_base.safetensors",
    f"{MODEL_BASE}/text_encoders")

download_model(REPO_ID,
    "split_files/vae/qwen_image_vae.safetensors",
    f"{MODEL_BASE}/vae")

# 保存先を確認
print("")
print("saved model files:")
for p in sorted(glob.glob(f"{MODEL_BASE}/**/*.safetensors", recursive=True)):
    print(" ", p)

print("")
print("model download complete")


downloading: anima-preview.safetensors ...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


split_files/diffusion_models/anima-previ(…):   0%|          | 0.00/4.18G [00:00<?, ?B/s]

done: anima-preview.safetensors -> /content/ComfyUI/models/diffusion_models/anima-preview.safetensors
downloading: qwen_3_06b_base.safetensors ...


split_files/text_encoders/qwen_3_06b_bas(…):   0%|          | 0.00/1.19G [00:00<?, ?B/s]

done: qwen_3_06b_base.safetensors -> /content/ComfyUI/models/text_encoders/qwen_3_06b_base.safetensors
downloading: qwen_image_vae.safetensors ...


split_files/vae/qwen_image_vae.safetenso(…):   0%|          | 0.00/254M [00:00<?, ?B/s]

done: qwen_image_vae.safetensors -> /content/ComfyUI/models/vae/qwen_image_vae.safetensors

saved model files:
  /content/ComfyUI/models/diffusion_models/anima-preview.safetensors
  /content/ComfyUI/models/text_encoders/qwen_3_06b_base.safetensors
  /content/ComfyUI/models/vae/qwen_image_vae.safetensors

model download complete


In [5]:
#@title オプション: その他のモデル（コメントアウトを外して使用）

# from huggingface_hub import hf_hub_download
# MODEL_BASE = f"{WORKSPACE}/models"

# ── SDXL ──
# hf_hub_download("stabilityai/stable-diffusion-xl-base-1.0",
#     "sd_xl_base_1.0.safetensors", local_dir=f"{MODEL_BASE}/checkpoints", local_dir_use_symlinks=False)
# hf_hub_download("stabilityai/stable-diffusion-xl-refiner-1.0",
#     "sd_xl_refiner_1.0.safetensors", local_dir=f"{MODEL_BASE}/checkpoints", local_dir_use_symlinks=False)

# ── FLUX.1 ──
# hf_hub_download("black-forest-labs/FLUX.1-schnell",
#     "flux1-schnell.safetensors", local_dir=f"{MODEL_BASE}/diffusion_models", local_dir_use_symlinks=False)

# ── VAE (汎用) ──
# hf_hub_download("stabilityai/sd-vae-ft-mse-original",
#     "vae-ft-mse-840000-ema-pruned.safetensors", local_dir=f"{MODEL_BASE}/vae", local_dir_use_symlinks=False)

# ── UpScale ──
# import urllib.request
# urllib.request.urlretrieve(
#     "https://github.com/xinntao/Real-ESRGAN/releases/download/v0.1.0/RealESRGAN_x4plus.pth",
#     f"{MODEL_BASE}/upscale_models/RealESRGAN_x4plus.pth")

print("オプションモデルセルです。必要なものをコメントアウト解除してください。")

オプションモデルセルです。必要なものをコメントアウト解除してください。


## Step 5: ComfyUI の起動

**cloudflared（推奨）** か **localtunnel** か **Colab iframe** の3種類から選んで実行してください。

In [ ]:
#@title 起動方法 A: cloudflared（推奨）

!wget -q -P ~ https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i ~/cloudflared-linux-amd64.deb

import subprocess
import threading
import time
import socket

def iframe_thread(port):
    while True:
        time.sleep(0.5)
        sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
        result = sock.connect_ex(('127.0.0.1', port))
        if result == 0:
            break
        sock.close()
    print("\nComfyUI の起動完了。cloudflared でトンネルを開きます...\n")
    p = subprocess.Popen(
        ["cloudflared", "tunnel", "--url", f"http://127.0.0.1:{port}"],
        stdout=subprocess.PIPE, stderr=subprocess.PIPE)
    for line in p.stderr:
        l = line.decode()
        if "trycloudflare.com " in l:
            print("🌐 ComfyUI アクセス URL:", l[l.find("http"):], end='')

threading.Thread(target=iframe_thread, daemon=True, args=(8188,)).start()

%cd /content/ComfyUI

# CPUで動かす場合はこちら（GPU制限解除待ちの場合）
# !python main.py --cpu --dont-print-server --listen --enable-cors-header *

# もしT4 GPUが使えるようになったら、--cpu を外して以下にしてください
# GPUで実行する場合
!python main.py --dont-print-server --listen --enable-cors-header '*'

(Reading database ... 122358 files and directories currently installed.)
Preparing to unpack .../cloudflared-linux-amd64.deb ...
Unpacking cloudflared (2026.3.0) over (2026.3.0) ...
Setting up cloudflared (2026.3.0) ...
Processing triggers for man-db (2.10.2-1) ...
/content/ComfyUI
[START] Security scan
INFO:root:[ComfyUI-Manager] Using `uv` as Python module for pip operations.
[ComfyUI-Manager] Using `uv` as Python module for pip operations.
Using Python 3.12.13 environment at: /usr
[DONE] Security scan
## ComfyUI-Manager: installing dependencies done.
** ComfyUI startup time: 2026-04-05 23:09:57.775
** Platform: Linux
** Python version: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
** Python executable: /usr/bin/python3
** ComfyUI Path: /content/ComfyUI
** ComfyUI Base Folder Path: /content/ComfyUI
** User directory: /content/ComfyUI/user
** ComfyUI-Manager config path: /content/ComfyUI/user/__manager/config.ini
** Log path: /content/ComfyUI/user/comfyui.log
Using Python 3.12.13

In [7]:
#@title 起動方法 B: localtunnel（cloudflared が使えない場合）

!npm install -g localtunnel

import subprocess
import threading
import time
import socket
import urllib.request

def iframe_thread(port):
    while True:
        time.sleep(0.5)
        sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
        result = sock.connect_ex(('127.0.0.1', port))
        if result == 0:
            break
        sock.close()
    print("\nComfyUI の起動完了。localtunnel でトンネルを開きます...\n")
    endpoint_ip = urllib.request.urlopen('https://ipv4.icanhazip.com').read().decode('utf8').strip()
    print("🔑 localtunnel パスワード / エンドポイント IP:", endpoint_ip)
    p = subprocess.Popen(["lt", "--port", str(port)], stdout=subprocess.PIPE)
    for line in p.stdout:
        print(line.decode(), end='')

threading.Thread(target=iframe_thread, daemon=True, args=(8188,)).start()

%cd /content/ComfyUI
!python main.py --dont-print-server

⠙⠹⠸⠼⠴⠦^C
/content/ComfyUI
[START] Security scan
INFO:root:[ComfyUI-Manager] Using `uv` as Python module for pip operations.
[ComfyUI-Manager] Using `uv` as Python module for pip operations.
Using Python 3.12.13 environment at: /usr
[DONE] Security scan
## ComfyUI-Manager: installing dependencies done.
** ComfyUI startup time: 2026-04-05 23:07:41.665
** Platform: Linux
** Python version: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
** Python executable: /usr/bin/python3
** ComfyUI Path: /content/ComfyUI
** ComfyUI Base Folder Path: /content/ComfyUI
** User directory: /content/ComfyUI/user
** ComfyUI-Manager config path: /content/ComfyUI/user/__manager/config.ini
** Log path: /content/ComfyUI/user/comfyui.log
Using Python 3.12.13 environment at: /usr
Using Python 3.12.13 environment at: /usr
INFO:root:
Prestartup times for custom nodes:

Prestartup times for custom nodes:
INFO:root:   2.9 seconds: /content/ComfyUI/custom_nodes/ComfyUI-Manager
   2.9 seconds: /content/ComfyUI/custom

In [8]:
#@title 起動方法 C: Colab iframe（WebSocket 非対応のため機能制限あり）

import threading
import time
import socket

def iframe_thread(port):
    while True:
        time.sleep(0.5)
        sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
        result = sock.connect_ex(('127.0.0.1', port))
        if result == 0:
            break
        sock.close()
    from google.colab import output
    output.serve_kernel_port_as_iframe(port, height=1024)
    print("別ウィンドウで開く場合はこちら:")
    output.serve_kernel_port_as_window(port)

threading.Thread(target=iframe_thread, daemon=True, args=(8188,)).start()

%cd /content/ComfyUI
!python main.py --dont-print-server

/content/ComfyUI
[START] Security scan
INFO:root:[ComfyUI-Manager] Using `uv` as Python module for pip operations.
[ComfyUI-Manager] Using `uv` as Python module for pip operations.
Using Python 3.12.13 environment at: /usr
[DONE] Security scan
## ComfyUI-Manager: installing dependencies done.
** ComfyUI startup time: 2026-04-05 23:07:55.683
** Platform: Linux
** Python version: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
** Python executable: /usr/bin/python3
** ComfyUI Path: /content/ComfyUI
** ComfyUI Base Folder Path: /content/ComfyUI
** User directory: /content/ComfyUI/user
** ComfyUI-Manager config path: /content/ComfyUI/user/__manager/config.ini
** Log path: /content/ComfyUI/user/comfyui.log
Using Python 3.12.13 environment at: /usr
Using Python 3.12.13 environment at: /usr
INFO:root:
Prestartup times for custom nodes:

Prestartup times for custom nodes:
INFO:root:   1.4 seconds: /content/ComfyUI/custom_nodes/ComfyUI-Manager
   1.4 seconds: /content/ComfyUI/custom_nodes/Co